# Code Review Agent Exploration

Run the cells from top to bottom. The final review cell requires `OPENAI_API_KEY` in `.env`.

## 1. Import the project code

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

load_dotenv(Path.cwd().parent / ".env")
MAX_CODE_LENGTH = 100_000
SYSTEM_PROMPT = """Review the code for bugs, security, performance, style, and improvements. Return a clear Markdown report with an overall rating."""
print("Notebook dependencies imported.")

Project code imported successfully.


In [5]:
# Confirm the key exists without printing its secret value.
status = 'configured' if os.getenv('OPENAI_API_KEY') else 'missing'
print(f'OPENAI_API_KEY: {status}')

OPENAI_API_KEY: configured


## 2. Validate source code

This is the first function we would later move into `agent.py`.

In [ ]:
def validate_code(code: str) -> str:
    """Trim code and reject empty or oversized input."""

    clean_code = code.strip()
    if not clean_code:
        raise ValueError("Code cannot be empty.")
    if len(clean_code) > MAX_CODE_LENGTH:
        raise ValueError("Code is too large to review.")
    return clean_code


sample_code = """
def divide(left, right):
    return left / right
"""

clean_code = validate_code(sample_code)
print(clean_code)
print(f"Characters: {len(clean_code)}")

def divide(left, right):
    return left / right
Characters: 48 / 100,000


In [ ]:
try:
    validate_code("   ")
except ValueError as error:
    print(f"Validation error: {error}")

Validation error: Code to review cannot be empty.


## 3. Build the review prompt

Inspect the prompt before sending code to the model.

In [ ]:
messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=f"Review this Python code:\n\n```python\n{clean_code}\n```")
]

print(messages[1].content)

Messages created: 2
Review this python code:

```python
def divide(left, right):
    return left / right
```


## 4. Read source from a file

This demonstrates file handling independently from the production module.

In [ ]:
example_path = Path("example_to_review.py")
example_path.write_text(clean_code, encoding="utf-8")
print(example_path.read_text(encoding="utf-8"))
example_path.unlink()

def divide(left, right):
    return left / right


## 5. Run the review

This final experiment calls OpenAI. It is the logic that can later become `review_code()`.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: OPENAI_API_KEY is not configured.")
else:
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    response = llm.invoke(messages)
    print(response.content)

# Code Review

## 1. Bugs & Correctness
- **Division by Zero**: The function does not handle the case where `right` is zero, which will raise a `ZeroDivisionError`. This should be handled to prevent the program from crashing.
- **Type Checking**: The function assumes that both `left` and `right` are numbers. If non-numeric types are passed, a `TypeError` will be raised. Consider adding type checks or using type hints to clarify expected input types.

## 2. Security Issues
- **Injection Risks**: There are no apparent injection risks in this simple function.
- **Secrets Exposure**: The function does not handle any sensitive data, so there are no concerns about secrets exposure.

## 3. Performance
- **Inefficiencies**: The function performs a single division operation, which is efficient.
- **Unnecessary Computation**: There are no unnecessary computations in this function.

## 4. Code Style
- **PEP 8 Violations**: The code is very short and adheres to PEP 8 guidelines.
- **Naming Convent

## 6. Run the Streamlit UI

The UI in `app.py` uses the same `review_code()` function. From Command Prompt in the agent folder, run:

```cmd
python -m streamlit run app.py
```